# ML Model Verification & Performance Analysis
This notebook verifies the performance of the trained ignition correction model located in `ML/run_2`.

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import pickle
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Paths
RUN_DIR = 'ML/run_2'
MODEL_PATH = os.path.join(RUN_DIR, 'correction_model.keras')
SCALER_PATH = os.path.join(RUN_DIR, 'correction_scalers.pkl')
DATA_PATH = os.path.join(RUN_DIR, 'correction_training_data_20260204_195041.csv')
FEATURES_PATH = os.path.join(RUN_DIR, 'feature_names.txt')

print(f"TensorFlow version: {tf.__version__}")

## 1. Data Loading and Feature Mapping

In [ ]:
# Load features
with open(FEATURES_PATH, 'r') as f:
    features = [line.strip() for line in f.readlines() if line.strip()]

print(f"Loaded {len(features)} feature names.")

# Load data
df = pd.read_csv(DATA_PATH)
X = df[features]
y = df['TARGET_correction']

print(f"Dataset shape: {df.shape}")
df.head()

## 2. Load Model and Scaler

In [ ]:
model = tf.keras.models.load_model(MODEL_PATH)
with open(SCALER_PATH, 'rb') as f:
    scaler = pickle.load(f)

print("Model Summary:")
model.summary()

## 3. Validation Performance (MAE Check)

In [ ]:
# Scale features
X_scaled = scaler.transform(X)

# Get predictions
y_pred = model.predict(X_scaled).flatten()

mae = mean_absolute_error(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))

print(f"Overall Mean Absolute Error: {mae:.4f} meters")
print(f"Overall RMSE: {rmse:.4f} meters")

plt.figure(figsize=(10, 6))
plt.scatter(y, y_pred, alpha=0.5)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2)
plt.xlabel('Ground Truth Correction (m)')
plt.ylabel('Model Predicted Correction (m)')
plt.title('Predicted vs Actual Ignition Correction')
plt.grid(True)
plt.show()

## 4. Sensitivity Analysis: Fault Impact
We check how the model adjusts the ignition altitude as faults increase.

In [ ]:
# Select a baseline sample (Medium rocket)
baseline_idx = df[df['rocket_type'] == 'Medium'].index[0]
sample = df.iloc[baseline_idx].copy()

mass_reductions = np.linspace(0, 5, 10) # 0 to 5kg loss
corrections = []

for m_loss in mass_reductions:
    temp_sample = sample.copy()
    # Inferred mass drops
    temp_sample['inferred_mass'] -= m_loss
    
    s_scaled = scaler.transform([temp_sample[features]])
    pred = model.predict(s_scaled, verbose=0)[0][0]
    corrections.append(pred)

plt.figure(figsize=(10, 5))
plt.plot(mass_reductions, corrections, 'o-')
plt.xlabel('Simulated Mass Loss (kg)')
plt.ylabel('Predicted Ignition Correction (m)')
plt.title('Sensitivity: Model Response to Mass Fault')
plt.grid(True)
plt.show()

## 5. Cross-Class Verification
Check MAE across different rocket types.

In [ ]:
df['pred'] = y_pred
df['error'] = np.abs(df['pred'] - df['TARGET_correction'])

class_metrics = df.groupby('rocket_type')['error'].agg(['mean', 'std', 'count'])
print("Performance by Rocket Class:")
print(class_metrics)

class_metrics['mean'].plot(kind='bar', yerr=class_metrics['std'], capsize=4, figsize=(10, 6))
plt.ylabel('Mean Absolute Error (m)')
plt.title('Prediction Error by Rocket Scale')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()